In [7]:
from test.helpers import SDE, simple_batch_sde_solve

import diffrax
import jax
import jax.numpy as jnp
import jax.random as jr
from diffrax import (
    ControlTerm,
    MultiTerm,
    ODETerm,
    ShARK,
    SpaceTimeLevyArea,
    VirtualBrownianTree,
)
from jax import Array


jax.config.update("jax_enable_x64", True)

vbt = VirtualBrownianTree(
    0.0,
    16.0,
    0.001,
    (),
    jr.key(8),
    SpaceTimeLevyArea,
    _w_t1=jnp.array(-3.0),
    _hh_t1=jnp.array(5.0),
)
out = vbt.evaluate(0.0, 3.0, use_levy=True)
print(out)
print(out.W)
print(out.H)

SpaceTimeLevyArea(dt=f64[], W=f64[], H=f64[])
4.127056709441923
-0.35470738388700485


In [2]:
# sst_sde is two-dimensional, the first dimension being W_t, the second \int_0^t W_s ds


def drift(t, y, args):
    return jnp.array([0.0, y[0] ** 2], dtype=y.dtype)


def diffusion(t, y, args):
    return jnp.array([1.0, 0.0], dtype=y.dtype)


def get_terms(bm):
    return MultiTerm(ODETerm(drift), ControlTerm(diffusion, bm))


y0 = jnp.array([0.0, 0.0], dtype=jnp.float64)
args = None
t0 = 0.0
t1 = 1.0
bm_shape = ()

sst_sde = SDE(get_terms, args, y0, t0, t1, bm_shape)

In [8]:
def compute_sst(w, hh, num_samples, key):
    keys = jr.split(key, num_samples)
    saveat = diffrax.SaveAt(t1=True)
    bm_tol = 2**-10
    dt0 = 2**-8
    solver = ShARK()
    controller = None
    levy_area = SpaceTimeLevyArea
    w = jnp.array(w, dtype=jnp.float64)
    hh = jnp.array(hh, dtype=jnp.float64)
    y1, _ = simple_batch_sde_solve(
        keys,
        sst_sde,
        solver,
        levy_area,
        dt0,
        controller,
        bm_tol,
        saveat,
        _w_t1=w,
        _hh_t1=hh,
    )
    sst = jnp.squeeze(y1)[:, 1]
    return sst


def true_cond_stats_c(w, hh):
    w2 = w**2
    hh2 = hh**2
    mean = 1 / 3 * w2 + w * hh + 6 / 5 * hh2 + 1 / 15
    var = 11 / 6300 + 1 / 180 * w2 + 1 / 175 * hh2
    return mean, var


def stat_error(c: Array, w, hh):
    w = float(w)
    hh = float(hh)
    true_mean, true_var = true_cond_stats_c(w, hh)
    mean_error = jnp.abs(true_mean - jnp.mean(c))
    var_error = jnp.abs(true_var - jnp.var(c))
    return mean_error, var_error

In [10]:
sst = compute_sst(3.0, 5.0, 2**18, jr.PRNGKey(0))
mean_error, var_error = stat_error(sst, 3.0, 5.0)
print(f"Mean error: {mean_error}, variance error: {var_error}")

(262144, 1, 2)
Mean error: 0.0018340642219740744, variance error: 0.00044919012939154124
